# 12 — The web search tool 

Important note: Your organization must enable the Web Search tool in the settings console before using it. You can find this setting here: https://console.anthropic.com/settings/privacy

Claude includes a built-in web search tool that lets it search the internet for current or specialized information to answer user questions. Unlike other tools where you need to provide the implementation, Claude handles the entire search process automatically - you just need to provide a simple schema to enable it.

### Setting Up the Web Search Tool
To use the web search tool, you create a schema object with these required fields:

In [1]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5
}

The max_uses field limits how many searches Claude can perform. Claude might do follow-up searches based on initial results, so this prevents excessive API calls. A single search returns multiple results, but Claude may decide additional searches are needed.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
from src.utils import add_assistant_message, add_user_message, chat
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [5]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise to gain leg muscle?
    """,
)
response = chat(
    messages=messages,
    client=client,
    model=model,
    tools=[web_search_schema],
)
response

Message(id='msg_01TA4HVUt2vS9AWbBNix22ch', container=None, content=[ServerToolUseBlock(id='srvtoolu_01WLg2xPtQEv1GnK91RVQFCF', caller=DirectCaller(type='direct'), input={'query': 'best exercises leg muscle growth hypertrophy'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EoEYCioIDhgCIiRmYzYzMzE0My1mMmFkLTRjMGEtYWU0Zi05NGQyYTdkMGEzY2USDAn9kM59YW+LGDOlgxoMQ9mAlRAFiiNSMBb0IjCltrxgQ4vX8prRHLIXyOFsgpIG7gwGJPi18rGxNxcKzNqX53yERq8AKZZQjPDoSc0qhBdXmufiTiQe22ieYN7Eclw5vHOgMUv3KPhVArJNFkemTNF5eaWBCRSWfXYrTftygQjmAB/cI3Qq+aVFBya6K+rXF35+mhCJuVZzCNrwatxOIRK9+rStV6PhYTCTG3J92FEjUSFQ99lXG0mBKhi1i9eaUM+oDpfD5Q2Thrq/eSRx4/pKAULUyHuyoHrEIsUSS8EGnacM388eI6ZT5SGuJXqk6Hth/vQ37VUWD6BqCMFkdudYqPWF2oe8OM7yBTj88S2hhxceyGo5c3+wCiO3d+TazQOS5cRv1h9BpLCUyM7uIlWXQicLfNQ6wtm7VoNl2gRfX8lmzoPXww+1Vsr0vqY1S69WmIV5d18SQ6ChrWyECMFKoPXug5Nhc/LrVl3RFkA5DyKBuvSvTOadk6J8cQWarYwPF5dc05uEstWUOod/3kyo2jKQ+hP8Ipua5dUQUF

### How the Response Works
When Claude uses the web search tool, the response contains several types of blocks:

- **Text blocks** - Claude's explanation of what it's doing
- **ServerToolUseBlock** - Shows the exact search query Claude used
- **WebSearchToolResultBlock** - Contains the search results
- **WebSearchResultBlock** - Individual search results with titles and URLs
- **Citation blocks** - Text that supports Claude's statements

The response structure lets you see exactly what Claude searched for and which sources it found. Citations include the specific text Claude used to support its answers, along with the source URLs.

### Restricting Search Domains
You can limit searches to specific domains using the allowed_domains field. This is particularly useful when you want reliable, authoritative sources:

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

For example, when asking about medical or exercise advice, restricting to domains like PubMed (nih.gov) ensures you get evidence-based information rather than random blog content.

In [4]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise to gain leg muscle?
    """,
)
response = chat(
    messages=messages,
    client=client,
    model=model,
    tools=[web_search_schema],
)
response

Message(id='msg_014sbuzFLMvV9PFcgi2N5QCX', container=None, content=[ServerToolUseBlock(id='srvtoolu_019DpUso4dwvc2WpxmunW7se', caller=DirectCaller(type='direct'), input={'query': 'best exercises build leg muscle mass'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='ErMlCioIDhgCIiRmYzYzMzE0My1mMmFkLTRjMGEtYWU0Zi05NGQyYTdkMGEzY2USDMAz7g7pgiN4hliHgxoMXM5b1F5hgd+BpgTQIjD/Ctib1UlneXH9yhGPi7MgS79N3ZS2QAwZzdgSfNpiwBeKpZXsTxLoyon0CNiR3mcqtiTNrCWA6rVWRj0EIeDz50kezpuaeXnJrvhGiIrpGm/Uc2Rrb8qw7KYwK/zdEvPQ6mSMQ6idZzyk4qFSgFesHhvHktdwVYQPXGq4vQhRff+/8Ts5GRhVS+q/JNDkwp6OXfvH5VmMe3tyMfS1PWOe2gmE/Hy1r597IbvuHL3dH0Sf9XJmlbM1uwUWH2JD90jliPlVPhp80QAVAyA6u3eG+e2Yr0r/PZs9YqbN9XxLyQAy6lyIu09RcQkuQWtyyQFDJ48NGhZTXi6YimAnawpHnUQYj8i17TCttEDWEy5HcfzITlrrTRX9K5Sy7fhLIy2rWNXPdameedWvp5Iyv6DG9ab9fY3DrVdzxQyE2vrwV+vTw3WhHkBAp41PC6gXsEnEg4AAfpeHf2/JQazjoqSHPHYEhwYrEoonzcIw5pLZt7JMDm3WDyb97Oxxg84oiqgtzGavR3TXxw